# 01 - Setup & smoke test (Kaggle version)

**Goal of this notebook:** get Kaggle ready to run TextCtrl, then prove it works on the repo's example images.
**Steps:** 0 CONFIG, 1 find datasets + check internet + GPU, 2 clone TextCtrl, 3 install packages, 3b import test,
4 Stable Diffusion weights, 5 link weights and check files.

### Before you run anything (Kaggle right-hand Settings panel)
1. **Accelerator:** choose **GPU T4 x2** (not P100: new torch builds may not support the older P100 chip).
2. **Internet:** turn **On** (needed to download TextCtrl, packages and Stable Diffusion). Kaggle asks you to verify your phone number first.
3. **Input:** click *Add Input* and add both of your datasets: **PATH_TO_DATA500_DATASET** and **weights**.

## Step 0 - CONFIG
Every notebook in this project starts with one settings cell, so all the important values (the seed, the folder
paths) live in ONE place and nothing is hidden inside later code. The seed is a starting number for the random
number generators: with the same seed, "random" choices come out the same every run, so a teammate (or your future
self) can repeat your results exactly. On Kaggle there are two kinds of folders: `/kaggle/input` (your uploaded
datasets, **read-only**) and `/kaggle/working` (where we may write). **Run this cell first, before Step 1.**

In [ ]:
# import random so we can fix Python's own random numbers
import random
# import numpy so we can fix its random numbers too
import numpy as np
# import torch so we can fix its random numbers too
import torch

# the seed: the same seed gives the same "random" results every run
SEED = 42
# the folder where Kaggle puts the datasets we attached (read-only)
INPUT_ROOT = "/kaggle/input"
# the folder where Kaggle mounts OUR datasets (it includes the Kaggle username)
DATASETS_ROOT = INPUT_ROOT + "/datasets/PATH_TO_KAGGLE_USERNAME"
# the PATH_TO_DATA500_DATASET dataset (500 sample pages)
DATA_ROOT = DATASETS_ROOT + "/PATH_TO_DATA500_DATASET"
# the weights dataset (the 7 .pth files; they may sit in an inner weights/ folder)
WEIGHTS_ROOT = DATASETS_ROOT + "/weights"
# the folder we are allowed to write to
WORK_DIR = "/kaggle/working"
# where the TextCtrl code gets cloned
TEXTCTRL_DIR = WORK_DIR + "/TextCtrl"
# the folder TextCtrl reads its weights from (7 .pth files + sd/)
WEIGHTS_DIR = TEXTCTRL_DIR + "/weights"
# the folder for the Stable Diffusion v1.5 parts
SD_DIR = WEIGHTS_DIR + "/sd"
# the GitHub address of TextCtrl
TEXTCTRL_URL = "https://github.com/weichaozeng/TextCtrl.git"
# the Hugging Face address of Stable Diffusion v1.5
SD_REPO = "stable-diffusion-v1-5/stable-diffusion-v1-5"

# fix Python's random generator
random.seed(SEED)
# fix numpy's random generator
np.random.seed(SEED)
# fix torch's random generator
torch.manual_seed(SEED)

# print a title for the settings
print("CONFIG loaded:")
# print the seed
print("  SEED         =", SEED)
# print the input folder
print("  INPUT_ROOT   =", INPUT_ROOT)
# print the PATH_TO_DATA500_DATASET dataset folder
print("  DATA_ROOT    =", DATA_ROOT)
# print the weights dataset folder
print("  WEIGHTS_ROOT =", WEIGHTS_ROOT)
# print the working folder
print("  WORK_DIR     =", WORK_DIR)
# print the TextCtrl code folder
print("  TEXTCTRL_DIR =", TEXTCTRL_DIR)
# print the weights folder
print("  WEIGHTS_DIR  =", WEIGHTS_DIR)
# print the Stable Diffusion folder
print("  SD_DIR       =", SD_DIR)
# print the Stable Diffusion source
print("  SD_REPO      =", SD_REPO)

## Step 1 - Find the datasets, check internet, check the GPU
Before installing anything we check three things. (1) Kaggle really attached our two datasets, and we find where
they are (the folder name can differ slightly from the dataset name). (2) Internet is on, because
downloads (git, pip, Hugging Face) need it. (3) We have a GPU.

**What is a GPU?** Think of the normal chip (the CPU) as one very smart accountant who does one calculation at a time.
A GPU is a stadium of thousands of simple clerks who each do a tiny sum at the same moment. TextCtrl repaints a
picture with millions of small multiplications, repeated 50 times, so a GPU does in seconds what a CPU does in minutes.
TextCtrl needs about 5-6 GB of GPU memory, and a T4 has about 15 GB, so it fits.

In [ ]:
# import sys so we can read the Python version
import sys
# import os so we can look inside folders
import os
# import subprocess so we can run the terminal command nvidia-smi from Python
import subprocess
# import urllib.request so we can test the internet connection
import urllib.request
# import torch, the library that does the GPU maths
import torch


# define a helper that looks for a folder under root (at most max_depth levels down)
def find_folder(root, max_depth, file_name=None, subfolder_suffix=None):
    # walk through every folder under root
    for folder, subfolders, files in os.walk(root):
        # work out how many levels below root this folder is
        depth = folder[len(root):].count(os.sep)
        # if we are looking for a file and this folder contains it, this is the folder
        if file_name is not None and file_name in files:
            # give back this folder
            return folder
        # if we are looking for a subfolder ending in a suffix, check the subfolders
        if subfolder_suffix is not None:
            # look at each subfolder name
            for sub in subfolders:
                # if the name ends with the suffix, this is the folder
                if sub.endswith(subfolder_suffix):
                    # give back this folder
                    return folder
        # do not walk deeper than max_depth (clearing the list stops os.walk from going down)
        if depth >= max_depth:
            # clear the list of subfolders
            subfolders[:] = []
    # nothing found
    return None


# print a header for the dataset check
print("=== Kaggle input datasets ===")
# check that the Kaggle input folder exists at all
if not os.path.isdir(INPUT_ROOT):
    # explain the problem
    print(f"{INPUT_ROOT} not found - are you running on Kaggle?")
    # there is nothing to find
    DATA_DIR = None
    # there is nothing to find
    PTH_DIR = None
# otherwise the input folder exists
else:
    # show the names directly inside /kaggle/input
    print("Inside /kaggle/input:", sorted(os.listdir(INPUT_ROOT))[:20])
    # check that the folder holding our datasets exists
    if os.path.isdir(DATASETS_ROOT):
        # show the names inside it (max 20)
        print("Inside", DATASETS_ROOT, ":", sorted(os.listdir(DATASETS_ROOT))[:20])
    # otherwise that folder does not exist
    else:
        # say so, because the Kaggle username in CONFIG may be wrong
        print("NOT FOUND:", DATASETS_ROOT, "- is the Kaggle username in CONFIG right?")
    # find the folder that holds the bank folders (names ending in _epdf), looking up to 3 levels inside PATH_TO_DATA500_DATASET
    DATA_DIR = find_folder(DATA_ROOT, 3, subfolder_suffix="_epdf")
    # if the pages are in a nested folder (e.g. an inner PATH_TO_DATA500_DATASET/), use it but say so
    if DATA_DIR is not None and DATA_DIR != DATA_ROOT:
        # tell the user we are using the inner folder
        print(f"  NOTE: pages are in a nested folder, using {DATA_DIR} (tell Claude)")
    # find the folder that holds model.pth, looking up to 2 levels inside the weights dataset
    PTH_DIR = find_folder(WEIGHTS_ROOT, 2, file_name="model.pth")
    # if the .pth files are in a nested folder, use it but say so
    if PTH_DIR is not None and PTH_DIR != WEIGHTS_ROOT:
        # tell the user we are using the inner folder
        print(f"  NOTE: .pth files are in a nested folder, using {PTH_DIR} (tell Claude)")

# if we found the pages folder
if DATA_DIR is not None:
    # start counting PNG pages
    n_pages = 0
    # walk through every folder inside the pages folder
    for folder, subfolders, files in os.walk(DATA_DIR):
        # look at each file name
        for file_name in files:
            # count it if it is a PNG
            if file_name.lower().endswith(".png"):
                # add one to the counter
                n_pages += 1
    # print where the pages are and how many there are (we expect 500)
    print(f"  found    pages folder: {DATA_DIR}  ({n_pages} PNG pages, expected 500)")
# otherwise the pages were not found
else:
    # tell the user what to check
    print(f"  MISSING  pages folder (looked inside {DATA_ROOT} for folders ending in _epdf). Did you Add Input > PATH_TO_DATA500_DATASET?")

# if we found the weights folder
if PTH_DIR is not None:
    # count the .pth files directly inside it
    n_pth = len([n for n in os.listdir(PTH_DIR) if n.endswith(".pth")])
    # print where the weights are and how many .pth files there are (we expect 7)
    print(f"  found    weights folder: {PTH_DIR}  ({n_pth} .pth files, expected 7)")
# otherwise the weights were not found
else:
    # tell the user what to check
    print(f"  MISSING  model.pth inside {WEIGHTS_ROOT}. Is the upload finished, and did you Add Input > weights?")

# print an empty line for spacing
print()
# print a header for the internet check
print("=== Internet ===")
# try to open a web page
try:
    # open Hugging Face's home page, giving up after 8 seconds
    urllib.request.urlopen("https://huggingface.co", timeout=8)
    # it worked
    print("Internet: ON")
# if opening failed for any reason
except Exception:
    # explain how to fix it
    print("Internet: OFF - Settings panel > Internet > On (needs phone verification), then restart the session.")

# print an empty line for spacing
print()
# print a header for the software check
print("=== Software ===")
# print the Python version (first word only, e.g. 3.12.x)
print("Python version :", sys.version.split()[0])
# print the torch version
print("torch version  :", torch.__version__)
# ask torch whether it can see a CUDA GPU (True / False)
cuda_ok = torch.cuda.is_available()
# print the answer
print("CUDA available :", cuda_ok)

# if a GPU is available, print its details
if cuda_ok:
    # get the name of GPU number 0
    gpu_name = torch.cuda.get_device_name(0)
    # get the properties (including total memory) of GPU number 0
    props = torch.cuda.get_device_properties(0)
    # convert the memory from bytes to gigabytes (1 GB = 1024**3 bytes)
    total_gb = props.total_memory / 1024**3
    # print the GPU name
    print("GPU name       :", gpu_name)
    # print the GPU memory with one decimal
    print(f"GPU memory     : {total_gb:.1f} GB")
    # check whether it looks like the T4 we expect
    if "T4" not in gpu_name:
        # warn that this is a different GPU (not a problem, just good to know)
        print("Note: this is not a T4, but it should still work.")
# otherwise there is no GPU
else:
    # explain how to fix it
    print("NO GPU FOUND. Go to Runtime > Change runtime type > T4 GPU, then run this cell again.")

# print an empty line for spacing
print()
# print a header for the nvidia-smi output
print("=== nvidia-smi (the GPU's own status report) ===")
# try to run nvidia-smi
try:
    # run the command and capture what it prints as text
    result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    # print what nvidia-smi said
    print(result.stdout)
# if the command does not exist, there is no GPU driver
except FileNotFoundError:
    # tell the user nvidia-smi is not available
    print("nvidia-smi not found (this means no GPU runtime is selected).")

## Step 2 - Get TextCtrl's code
So far we only checked the machine (Step 1). Now we download the TextCtrl program itself from GitHub, like
downloading a recipe book. The recipes are the Python files; the ingredients (the trained weights) come later.
Kaggle starts every session with a clean `/kaggle/working`, so this cell clones only if the folder is not already there.

In [ ]:
# import os so we can check and list folders
import os
# import subprocess so we can run git from Python
import subprocess

# check whether the repo is already cloned (a .git folder exists inside it)
if os.path.isdir(TEXTCTRL_DIR + "/.git"):
    # say we are skipping the download
    print("TextCtrl is already cloned, skipping.")
# otherwise we need to clone it
else:
    # clone only the newest snapshot (--depth 1 means no history, so it is fast)
    result = subprocess.run(["git", "clone", "--depth", "1", TEXTCTRL_URL, TEXTCTRL_DIR], capture_output=True, text=True)
    # print the end of git's messages
    print(result.stdout[-500:], result.stderr[-500:])
    # if git failed (code is not 0), stop with a clear message
    if result.returncode != 0:
        # raise an error so the notebook stops here
        raise RuntimeError("git clone failed - read the message above")

# list what is inside the cloned folder, in alphabetical order
names = sorted(os.listdir(TEXTCTRL_DIR))
# print how many items there are
print(f"{len(names)} items in {TEXTCTRL_DIR}")
# print the first 20 names only
print(names[:20])
# ask git which exact version (commit) we got
commit = subprocess.run(["git", "-C", TEXTCTRL_DIR, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
# print the commit so we can reproduce this exact code later
print("commit:", commit)

## Step 3 - Install only what is missing
TextCtrl ships a shopping list (`requirement.txt`) written for Python 3.8 and torch 1.13 (2022-2023). Kaggle today
have a much newer Python and torch. Buying everything on the old list would overwrite things Kaggle already has and break
them. So we do it like checking the fridge first: for each package, if Python can already find it we skip it,
otherwise we install the **newest** version (no old pin). **torch is never touched.**
Each package prints one plain-English line saying what it is for. If one fails we stop and fix that one first.

In [ ]:
# import sys so we know which Python is running
import sys
# import subprocess so we can run pip from Python
import subprocess
# import importlib.util so we can ask "is this package installed?"
import importlib.util
# import importlib so we can refresh Python's package list after installing
import importlib

# make an empty list that will collect (package, result) pairs for the summary
results = []

# define a helper that installs one package only if Python cannot find it
def install_if_missing(pip_name, import_name, eli5):
    # tell the user what this package is, in one simple line
    print(f"- {pip_name}: {eli5}")
    # ask Python whether it can already find the package
    already_there = importlib.util.find_spec(import_name) is not None
    # if it is already installed, we do nothing
    if already_there:
        # report that we skipped it
        print("    already installed, skipped")
        # remember the result for the summary
        results.append((pip_name, "already installed"))
        # stop here for this package
        return
    # otherwise run pip to install it, capturing the messages pip prints
    run = subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", pip_name], capture_output=True, text=True)
    # if pip finished with code 0 it worked
    if run.returncode == 0:
        # report success
        print("    installed OK")
        # remember the result for the summary
        results.append((pip_name, "installed now"))
    # otherwise pip failed
    else:
        # say it failed
        print("    FAILED. Last lines of pip's message:")
        # show only the last 15 lines of the error to keep the output short
        print("\n".join(run.stderr.strip().split("\n")[-15:]))
        # remember the failure for the summary
        results.append((pip_name, "FAILED"))

# download tool for Hugging Face (needed in Step 4)
install_if_missing("huggingface_hub", "huggingface_hub", "lets Python download files from the Hugging Face website")
# safe file format for model weights
install_if_missing("safetensors", "safetensors", "a safe, fast file format for saved model weights (the Stable Diffusion files use it)")
# library of ready-made models
install_if_missing("transformers", "transformers", "a big shelf of ready-made text and vision models (TextCtrl's requirement list includes it)")
# helper for loading big models
install_if_missing("accelerate", "accelerate", "helps load big models onto the GPU smoothly (in TextCtrl's requirement list)")
# readable tensor reshaping
install_if_missing("einops", "einops", "reshapes big grids of numbers (tensors) with short, readable commands")
# settings-file reader
install_if_missing("omegaconf", "omegaconf", "reads settings files (.yaml) and turns them into easy-to-use objects")
# torch wrapper
install_if_missing("pytorch-lightning", "pytorch_lightning", "a tidy wrapper around torch; TextCtrl's inference.py uses its seed_everything function")
# perceptual image distance
install_if_missing("lpips", "lpips", "a ruler that measures how different two images LOOK to a neural network")
# image-model shelf
install_if_missing("timm", "timm", "a shelf of ready-made image models (the text-reading ViTSTR model is built from it)")
# natural file sorting
install_if_missing("natsort", "natsort", "sorts file names the human way: img2 before img10")

# make Python notice the packages we just installed
importlib.invalidate_caches()
# print an empty line for spacing
print()
# print a title for the summary table
print("=== Summary ===")
# loop over the collected results
for pip_name, status in results:
    # print one row per package
    print(f"  {pip_name:18} {status}")

### Step 3b - Does TextCtrl's own code import?
Installing packages is like buying ingredients; this cell checks the recipe actually starts. We try to `import` the
exact modules that `inference.py` uses. We try ALL of them and show the end of each error (several may share one cause). The usual fix is one more missing
package or one old-code incompatibility, and we fix those one at a time (and log each in `notes/decisions.md`).
If this cell prints an error, copy the last ~30 lines to Claude.

**Fix found here (error 1):** TextCtrl ships its OWN old copy of `diffusers`, and it asks `huggingface_hub` for two
tools (`HfFolder`, `cached_download`) that new versions removed. The small "shim" block below adds those two names
back at runtime, so we do not have to edit TextCtrl's files. Any later notebook that imports TextCtrl needs the same block.

**Fix found here (error 2):** the old `diffusers` also loads its JAX/Flax parts when `jax` and `flax` are installed (Kaggle and Colab have them), and new JAX removed `jax.random.KeyArray`. TextCtrl runs on torch and never uses JAX, so the cell sets the old library's own switch `USE_FLAX=0`. It also makes Python forget any half-loaded copy of `diffusers` left over from an earlier failed run (otherwise the new setting is ignored until you restart the runtime).

**Fix found here (error 3):** the old `diffusers` asks `transformers` for `CLIPFeatureExtractor`, which was renamed to `CLIPImageProcessor`. The cell adds the old name back as another name for the new class.

**Fix found here (error 4):** in a FRESH Kaggle session the first import pass failed on `CLIPFeatureExtractor` even though
the shim had run, while a second pass in the same session worked. The cell now tries the whole import test up to two
times, re-applying the CLIP name to whatever `transformers` module Python is using at that moment, and prints whether
that module was swapped. A failed attempt 1 followed by `All imports OK` on attempt 2 is fine.

In [ ]:
# import sys so we can tell Python where TextCtrl's code is
import sys
# import os so we can move into the repo folder
import os
# import importlib so we can import modules by name
import importlib
# import traceback so we can print the error details if an import fails
import traceback

# let Python find TextCtrl's modules (src/ and utils.py)
sys.path.insert(0, TEXTCTRL_DIR)
# move into the repo folder, because its scripts use relative paths like weights/model.pth
os.chdir(TEXTCTRL_DIR)

# switch off the Flax/JAX parts of TextCtrl's old diffusers (its own official switch; TextCtrl uses torch, and new JAX broke those parts)
os.environ["USE_FLAX"] = "0"
# loop over a copy of the names of every module Python has already loaded
for loaded_name in list(sys.modules):
    # pick out modules that belong to TextCtrl's bundled diffusers or its src package
    if loaded_name == "diffusers" or loaded_name.startswith("diffusers.") or loaded_name.startswith("src"):
        # forget them, so they are re-imported with the new setting instead of reusing a half-loaded copy from an earlier failed run
        del sys.modules[loaded_name]

# import huggingface_hub so we can add two old names that TextCtrl's bundled diffusers still asks for
import huggingface_hub

# define a stand-in for HfFolder, an old helper that new huggingface_hub versions removed
class HfFolderShim:
    # the old code only calls HfFolder.get_token(), so that is the only thing we provide
    @staticmethod
    # define get_token
    def get_token():
        # return the saved Hugging Face login token (or None if we are not logged in)
        return huggingface_hub.get_token()

# only add the stand-in if the real HfFolder is gone
if not hasattr(huggingface_hub, "HfFolder"):
    # add our stand-in under the old name
    huggingface_hub.HfFolder = HfFolderShim
# only add this if the old cached_download is gone
if not hasattr(huggingface_hub, "cached_download"):
    # the old cached_download is replaced by hf_hub_download in new versions
    huggingface_hub.cached_download = huggingface_hub.hf_hub_download

# define a stand-in for Repository, an old helper for pushing models to Hugging Face (we never push)
class RepositoryShim:
    # if anything ever tries to create one, stop with a clear message
    def __init__(self, *args, **kwargs):
        # explain why this is not available
        raise RuntimeError("Repository (pushing to Hugging Face) is not available and not needed for inference")

# only add the stand-in if the real Repository is gone
if not hasattr(huggingface_hub, "Repository"):
    # add our stand-in under the old name
    huggingface_hub.Repository = RepositoryShim

# import the place where the old code looks for three error classes
import huggingface_hub.utils as hub_utils
# import the place where new versions keep those error classes
import huggingface_hub.errors as hub_errors
# loop over the three error class names the old code imports
for error_name in ["EntryNotFoundError", "RepositoryNotFoundError", "RevisionNotFoundError"]:
    # if the old location no longer has this name
    if not hasattr(hub_utils, error_name):
        # copy it over from the new location
        setattr(hub_utils, error_name, getattr(hub_errors, error_name))

# import transformers so we can add one more old name that TextCtrl's bundled diffusers asks for
import transformers
# always store the old name CLIPFeatureExtractor directly on the module (it was renamed to CLIPImageProcessor; same job)
transformers.CLIPFeatureExtractor = transformers.CLIPImageProcessor

# print a title for the shim report (if you do NOT see this line, you are running an OLD copy of this cell)
print("=== Step 3b shim report (cell version 5) ===")
# print the transformers version
print("transformers version      :", transformers.__version__)
# print the huggingface_hub version
print("huggingface_hub version   :", huggingface_hub.__version__)
# print whether HfFolder is now available
print("HfFolder available        :", hasattr(huggingface_hub, "HfFolder"))
# print whether Repository is now available
print("Repository available      :", hasattr(huggingface_hub, "Repository"))
# print whether CLIPFeatureExtractor is now available
print("CLIPFeatureExtractor      :", hasattr(transformers, "CLIPFeatureExtractor"))
# print the Flax switch
print("USE_FLAX                  :", os.environ.get("USE_FLAX"))
# print an empty line for spacing
print()

# the modules that inference.py imports, in the order it imports them
modules_to_test = ["pytorch_lightning", "src.trainer.CtrlBase", "src.MuSA.GaMuSA_app", "src.MuSA.GaMuSA", "utils"]
# remember whether everything worked
all_ok = False
# run the whole import test up to two times (in a fresh session the first pass can fail, see the markdown above)
for attempt in [1, 2]:
    # get the transformers module Python is using RIGHT NOW (it may have been swapped during an earlier import)
    current_transformers = sys.modules["transformers"]
    # store the old CLIP name on it again
    current_transformers.CLIPFeatureExtractor = current_transformers.CLIPImageProcessor
    # loop over a copy of the names of every loaded module
    for loaded_name in list(sys.modules):
        # pick out TextCtrl's bundled diffusers and its src package
        if loaded_name == "diffusers" or loaded_name.startswith("diffusers.") or loaded_name.startswith("src"):
            # forget them, so this attempt imports them cleanly
            del sys.modules[loaded_name]
    # assume this attempt works until an import fails
    attempt_ok = True
    # print which attempt this is
    print(f"--- import attempt {attempt} ---")
    # try each module one by one
    for module_name in modules_to_test:
        # try to import it
        try:
            # do the import
            importlib.import_module(module_name)
            # report success
            print(f"  OK      {module_name}")
        # if anything goes wrong, handle it
        except Exception:
            # report the failure
            print(f"  FAILED  {module_name}")
            # show only the last 8 lines of the error
            print("\n".join(traceback.format_exc().strip().split("\n")[-8:]))
            # remember that this attempt failed
            attempt_ok = False
    # report whether the transformers module was swapped for a different object (our diagnosis)
    print("transformers module swapped during import:", sys.modules["transformers"] is not transformers)
    # if every import worked, we are done
    if attempt_ok:
        # remember success
        all_ok = True
        # stop trying
        break

# print the final verdict
print("All imports OK" if all_ok else "Some imports failed - send the messages above to Claude")

## Step 4 - Stable Diffusion v1.5 weights (download only what is missing)
TextCtrl is built on top of Stable Diffusion, the general-purpose "painter": `vae/` zips and unzips images, `unet/`
is the painter's hand, `scheduler/` is the step-by-step recipe. They are not in your Kaggle datasets, so we download
them from Hugging Face into `TextCtrl/weights/sd/` (inside `/kaggle/working`, because `/kaggle/input` is read-only).
The repo stores each model in several formats (`.bin`, `.safetensors`, `fp16`, `non_ema`); we take the **`.bin`** file per folder (TextCtrl's config names the `.bin` files, and the old `diffusers` copy bundled with it cannot read `.safetensors`)
plus its small config, about 3.8 GB instead of ~20 GB. The cell checks every file first and skips what is already there.

In [ ]:
# import os so we can check files and make folders
import os
# import time so we can time the download
import time
# import the Hugging Face download function
from huggingface_hub import snapshot_download

# the files we need from vae/ (settings + the .bin weights file that TextCtrl asks for)
vae_files = ["vae/config.json", "vae/diffusion_pytorch_model.bin"]
# the files we need from unet/ (settings + the big .bin weights file that TextCtrl asks for)
unet_files = ["unet/config.json", "unet/diffusion_pytorch_model.bin"]
# the file we need from scheduler/ (just the settings)
scheduler_files = ["scheduler/scheduler_config.json"]
# put all wanted files in one list
wanted = vae_files + unet_files + scheduler_files

# make sure the sd/ folder exists
os.makedirs(SD_DIR, exist_ok=True)
# start an empty list for files that are not here yet
missing = []
# check every wanted file
for rel_path in wanted:
    # build the full path of this file
    full_path = os.path.join(SD_DIR, rel_path)
    # a file counts as present only if it exists and is not empty
    present = os.path.isfile(full_path) and os.path.getsize(full_path) > 0
    # if it is not present, add it to the missing list
    if not present:
        # remember it
        missing.append(rel_path)

# print how many files we need and how many are missing
print(f"Need {len(wanted)} files, missing {len(missing)}")
# if nothing is missing we can skip the download
if len(missing) == 0:
    # say so
    print("Everything is already here - no download.")
# otherwise download the missing files
else:
    # print which files will be downloaded
    print("Downloading:", missing)
    # note the start time
    start = time.time()
    # download only the missing files from Hugging Face straight into the sd/ folder
    snapshot_download(repo_id=SD_REPO, local_dir=SD_DIR, allow_patterns=missing)
    # work out how long it took
    seconds = time.time() - start
    # print the time
    print(f"Download finished in {seconds:.0f} seconds")

# print an empty line for spacing
print()
# print a title for the file check
print("=== What is in weights/sd/ now ===")
# check every wanted file again
for rel_path in wanted:
    # build the full path
    full_path = os.path.join(SD_DIR, rel_path)
    # if the file exists, print its size in megabytes
    if os.path.isfile(full_path):
        # convert bytes to megabytes (1 MB = 1024 * 1024 bytes)
        size_mb = os.path.getsize(full_path) / (1024 * 1024)
        # print one row
        print(f"  OK       {rel_path:50} {size_mb:8.1f} MB")
    # otherwise it is still missing
    else:
        # print a warning row
        print(f"  MISSING  {rel_path}")

## Step 5 - Put the 7 .pth files next to sd/ and check everything
TextCtrl looks for its files at `TextCtrl/weights/...` (the 7 `.pth` files directly inside, `sd/` as a subfolder).
Our `.pth` files live in the read-only `weights` dataset, so we make a **symbolic link** (a desktop-style shortcut)
for each one inside `TextCtrl/weights/`. Nothing is copied, so no extra disk space is used. Then we list the files so
you can tick off the 7 `.pth` files and the Stable Diffusion parts with your own eyes.

In [ ]:
# import os so we can make links and list files
import os

# the 7 weight files TextCtrl's README says must be directly inside weights/
expected_pth = ["model.pth", "text_encoder.pth", "style_encoder.pth", "vision_model.pth", "ocr_model.pth", "vgg19.pth", "vitstr_base_patch16_224.pth"]

# stop early if Step 1 did not find the weights dataset
if PTH_DIR is None:
    # stop with a clear message
    raise RuntimeError("PTH_DIR is None: the weights dataset was not found in Step 1. Check the upload and Add Input.")

# make sure the weights folder exists (Step 4 already created it together with sd/)
os.makedirs(WEIGHTS_DIR, exist_ok=True)
# make one shortcut per .pth file
for name in expected_pth:
    # where the real file lives (read-only dataset)
    real_file = os.path.join(PTH_DIR, name)
    # where TextCtrl will look for it
    link_file = os.path.join(WEIGHTS_DIR, name)
    # if the real file is missing, say so and move on
    if not os.path.isfile(real_file):
        # print a warning
        print(f"  not in dataset: {name}")
        # go to the next file
        continue
    # if an old shortcut is there, remove only the shortcut (no data is deleted)
    if os.path.islink(link_file):
        # remove the old shortcut
        os.remove(link_file)
    # if nothing is there now, make the shortcut
    if not os.path.exists(link_file):
        # create the shortcut
        os.symlink(real_file, link_file)

# print an empty line for spacing
print()
# print a title for the .pth check
print("=== The 7 .pth files (seen from TextCtrl/weights/) ===")
# count how many we find
found_count = 0
# check each expected file
for name in expected_pth:
    # build its path inside TextCtrl/weights/
    file_path = os.path.join(WEIGHTS_DIR, name)
    # if the file can be opened through the shortcut, print its size
    if os.path.isfile(file_path):
        # convert bytes to megabytes
        size_mb = os.path.getsize(file_path) / (1024 * 1024)
        # print one row
        print(f"  OK       {name:32} {size_mb:9.1f} MB")
        # count it
        found_count += 1
    # otherwise it is missing
    else:
        # print a warning row
        print(f"  MISSING  {name}")

# print an empty line for spacing
print()
# print a title for the sd check
print("=== Stable Diffusion parts (seen from TextCtrl/weights/) ===")
# count the SD files we find
sd_found = 0
# check each wanted SD file (the list from Step 4)
for rel_path in wanted:
    # build its path inside TextCtrl/weights/sd/
    file_path = os.path.join(SD_DIR, rel_path)
    # if it exists, count it and print it
    if os.path.isfile(file_path):
        # count it
        sd_found += 1
        # print one row
        print(f"  OK       sd/{rel_path}")
    # otherwise it is missing
    else:
        # print a warning row
        print(f"  MISSING  sd/{rel_path}")

# print an empty line for spacing
print()
# list the first 20 names directly inside weights/ to spot extra or nested folders
print("Everything directly inside TextCtrl/weights/ (max 20):", sorted(os.listdir(WEIGHTS_DIR))[:20])
# print the final verdict
print(f"\nVERDICT: {found_count}/7 .pth files, {sd_found}/{len(wanted)} Stable Diffusion files")

## Step 6 - Four fixes found by reading TextCtrl's code BEFORE running it
Reading `inference.py`, `utils.py` and the config ahead of time showed four things that would break on today's
Python 3.13 / torch 2.11. Fixing them now saves four failed runs. None of them edits a TextCtrl file.

1. **`torch.load` is stricter now.** Since torch 2.6 it refuses to open older checkpoint files unless told it is
   safe (`weights_only=False`). TextCtrl calls it in 6 places. We set the old default for this notebook session. This
   is safe here because the only files opened are the authors' `.pth` files and Stable Diffusion's own weights, not files from strangers.
2. **`np.int` no longer exists.** One TextCtrl file still writes `np.int`, which numpy deleted. We add the old names
   back as plain aliases (`np.int = int`).
3. **A font file is missing.** The config names `weights/arial.ttf`, which is not in your dataset. It is probably
   only used for training-log pictures, but as cheap insurance we link a look-alike font (Liberation Sans is made to
   match Arial's letter sizes).
4. **Pre-flight check.** We list every file the config names and confirm each one exists, so a missing file shows up
   here as one clear line instead of as a long error later.

**Expected output:** three short "done" lines, then a list of `OK` file checks ending with `PRE-FLIGHT: READY`.
If you restart the Kaggle session, run Step 3b again too, because these fixes live only in the running session.

In [ ]:
# import os so we can check files and make links
import os
# import numpy so we can add back its old names
import numpy as np
# import torch so we can change how torch.load behaves
import torch

# ---------- fix 1: torch.load keeps its OLD default (weights_only=False) ----------
# only wrap torch.load once, even if this cell is run twice
if not getattr(torch.load, "textctrl_patched", False):
    # remember the real torch.load
    original_torch_load = torch.load
    # define our version, which accepts the same inputs
    def torch_load_old_default(*args, **kwargs):
        # if the caller did not choose weights_only, choose False (the old behaviour)
        kwargs.setdefault("weights_only", False)
        # call the real torch.load with the settings
        return original_torch_load(*args, **kwargs)
    # leave a label so we know it is already wrapped
    torch_load_old_default.textctrl_patched = True
    # replace torch.load with our version
    torch.load = torch_load_old_default
# report that fix 1 is done
print("fix 1 done: torch.load uses weights_only=False (trusted files only)")

# ---------- fix 2: give numpy back its removed names ----------
# go through each removed name and the plain Python type it used to mean
for alias_name, python_type in [("int", int), ("float", float), ("bool", bool), ("object", object)]:
    # only add the alias if numpy does not have it
    if not hasattr(np, alias_name):
        # add the old name back
        setattr(np, alias_name, python_type)
# report that fix 2 is done
print("fix 2 done: np.int / np.float / np.bool / np.object aliases available")

# ---------- fix 3: a stand-in for weights/arial.ttf ----------
# where TextCtrl's config expects the font
font_link = WEIGHTS_DIR + "/arial.ttf"
# if a font is already there, leave it alone
if os.path.exists(font_link):
    # report it
    print("fix 3 done: weights/arial.ttf already exists")
# otherwise look for a look-alike font on this machine
else:
    # the fonts we would accept, best first
    candidates = ["/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf", "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"]
    # nothing chosen yet
    chosen_font = None
    # check each candidate
    for candidate in candidates:
        # take the first one that exists
        if os.path.isfile(candidate) and chosen_font is None:
            # remember it
            chosen_font = candidate
    # if we found one, link it under the name TextCtrl expects
    if chosen_font is not None:
        # make the shortcut
        os.symlink(chosen_font, font_link)
        # report which font stands in
        print("fix 3 done: weights/arial.ttf now points to", chosen_font)
    # otherwise there is no stand-in
    else:
        # say so; it only matters for training-log pictures
        print("fix 3 skipped: no look-alike font found (probably only needed for training-log pictures)")

# ---------- fix 4: pre-flight check of every file the config names ----------
# print an empty line for spacing
print()
# print a title for the check
print("=== Pre-flight check ===")
# the files inference.py and configs/inference.yaml need
needed_files = [
    TEXTCTRL_DIR + "/configs/inference.yaml",
    TEXTCTRL_DIR + "/example/i_s.txt",
    TEXTCTRL_DIR + "/example/i_t.txt",
    WEIGHTS_DIR + "/model.pth",
    WEIGHTS_DIR + "/text_encoder.pth",
    WEIGHTS_DIR + "/ocr_model.pth",
    WEIGHTS_DIR + "/vision_model.pth",
    WEIGHTS_DIR + "/sd/vae/config.json",
    WEIGHTS_DIR + "/sd/vae/diffusion_pytorch_model.bin",
    WEIGHTS_DIR + "/sd/unet/diffusion_pytorch_model.bin",
    WEIGHTS_DIR + "/sd/scheduler/scheduler_config.json",
]
# assume everything is fine until a file is missing
all_present = True
# check each file
for needed in needed_files:
    # if it exists, print OK
    if os.path.isfile(needed):
        # print the short name
        print("  OK       " + needed.replace(TEXTCTRL_DIR + "/", ""))
    # otherwise print MISSING
    else:
        # print the short name
        print("  MISSING  " + needed.replace(TEXTCTRL_DIR + "/", ""))
        # remember that something is missing
        all_present = False
# print the verdict
print("PRE-FLIGHT: READY" if all_present else "PRE-FLIGHT: NOT READY - if the .bin files are missing, re-run Step 4 first")

## Step 7 - Smoke test: run TextCtrl on the repo's own example images
This is the moment of truth for the setup. Everything so far (GPU, packages, weights, fixes) was preparation; now we
actually repaint pictures. The repo ships a tiny `example/` folder in exactly the format we will use later:
- `i_s/` = the **s**ource images: small crops, each showing one word.
- `i_s.txt` = one line per crop, `filename word`: the word each crop shows NOW.
- `i_t.txt` = one line per crop, `filename word`: the **t**arget word we WANT it to show instead.

TextCtrl splits each line on a single space, which is why words may not contain spaces. If an edited crop shows the
target word in the source crop's font, colour and background, the whole machine works.

**What the code does:** it runs the repo's own `inference.py` (default settings: 50 steps, seed 42) inside this
notebook, so the fixes from Steps 3b and 6 are active. It times the run, saves the edited crops to
`/kaggle/working/runs/smoke_test/`, and draws a grid with one row per example: source crop | source -> target text | TextCtrl result.

**Expected output:**
- a progress bar (one step per example crop); the very first crop takes longest because the model is loading;
- `Images edited: N / N`, total time, **seconds per crop**, peak GPU memory (the authors say about 5-6 GB);
- `SMOKE TEST PASSED` and the grid, saved as `runs/smoke_test/smoke_test_grid.png`.

**Write down (for the slides):** seconds per crop, one example that looks good, one that looks off.
Seconds per crop here includes loading the model, so it over-estimates; the real speed comes from the pilot in notebook 02.
**If it fails:** the cell prints the last 15 lines of the error. Copy those lines to Claude.

In [ ]:
# import os so we can work with folders
import os
# import sys so Python can find TextCtrl's code
import sys
# import time so we can time the run
import time
# import traceback so a failure prints a short message
import traceback
# import torch so we can read the GPU memory used
import torch
# import matplotlib so we can draw the before/after picture
import matplotlib.pyplot as plt
# import PIL so we can open the images
from PIL import Image

# the settings for this smoke test
NUM_STEPS = 50
# the repo's example folder (contains i_s/, i_s.txt, i_t.txt)
EXAMPLE_DIR = TEXTCTRL_DIR + "/example"
# where the edited images will be saved (the plan's runs/smoke_test/ folder)
SMOKE_OUT = WORK_DIR + "/runs/smoke_test"
# where the grid picture will be saved
FIGURE_PATH = SMOKE_OUT + "/smoke_test_grid.png"

# move into the repo folder, because inference.py uses relative paths like weights/model.pth
os.chdir(TEXTCTRL_DIR)
# make sure Python can import inference.py
if TEXTCTRL_DIR not in sys.path:
    # add the repo folder
    sys.path.insert(0, TEXTCTRL_DIR)


# define a helper that reads a "filename text" file into a dictionary (split on ONE space, like TextCtrl does)
def read_pairs(txt_path):
    # start with an empty dictionary
    pairs = {}
    # open the text file
    with open(txt_path) as txt_file:
        # go through each line
        for line in txt_file.readlines():
            # ignore empty lines
            if line.strip() != "":
                # split the line into the file name and the word
                image_name, word = line.strip().split(" ")
                # store it
                pairs[image_name] = word
    # give back the dictionary
    return pairs


# read what each image shows now
source_words = read_pairs(EXAMPLE_DIR + "/i_s.txt")
# read what we want each image to show
target_words = read_pairs(EXAMPLE_DIR + "/i_t.txt")
# list the example images in order
input_names = sorted(os.listdir(EXAMPLE_DIR + "/i_s"))
# print how many example images there are
print(f"Example images: {len(input_names)}")

# reset the GPU memory counter so we measure only this run
torch.cuda.reset_peak_memory_stats()
# assume the run failed until it succeeds
run_ok = False
# start the stopwatch
start = time.time()
# try to run TextCtrl
try:
    # import the repo's inference.py (this does not run it yet)
    import inference
    # build the default options exactly like the command line would
    opt = inference.create_parser().parse_args([])
    # point it at the example folder
    opt.dataset_dir = EXAMPLE_DIR + "/"
    # tell it where to save the edited images
    opt.output_dir = SMOKE_OUT
    # use our seed
    opt.seed = SEED
    # use 50 diffusion steps (the repo default); as a number, not text
    opt.num_inference_steps = NUM_STEPS
    # tell the user what happens next
    print("Running TextCtrl - the model loads first, so the progress bar may take a while to start...")
    # run the repo's main function
    inference.main(opt)
    # the run finished without an error
    run_ok = True
# if anything went wrong
except Exception:
    # print only the last 15 lines of the error
    print("\n".join(traceback.format_exc().strip().split("\n")[-15:]))
# stop the stopwatch
seconds = time.time() - start

# only report results if the run worked
if run_ok:
    # list the edited images that were written
    made = sorted([n for n in os.listdir(SMOKE_OUT) if n.lower().endswith(".png")])
    # read the peak GPU memory in gigabytes
    peak_gb = torch.cuda.max_memory_allocated() / 1024**3
    # work out the average seconds per image (includes loading the model)
    per_image = seconds / max(len(made), 1)
    # print an empty line for spacing
    print()
    # print a title for the summary
    print("=== Smoke test summary ===")
    # print how many images were edited
    print(f"Images edited        : {len(made)} / {len(input_names)}")
    # print the total time
    print(f"Total time           : {seconds:.0f} s (includes loading the model)")
    # print seconds per image
    print(f"Seconds per crop     : {per_image:.1f} s (over-estimate)")
    # print a rough estimate for 50 crops
    print(f"Rough time for 50    : {per_image * 50 / 60:.1f} minutes (over-estimate)")
    # print the GPU memory
    print(f"Peak GPU memory      : {peak_gb:.1f} GB")
    # print the verdict
    print("SMOKE TEST PASSED" if len(made) == len(input_names) else "SMOKE TEST INCOMPLETE - some images were not written")

    # one row per example, three columns: source crop | source -> target text | TextCtrl result
    fig, axes = plt.subplots(len(made), 3, figsize=(8, 1.3 * len(made)), squeeze=False)
    # fill one row per example image
    for row, name in enumerate(made):
        # open the source crop
        source_img = Image.open(EXAMPLE_DIR + "/i_s/" + name).convert("RGB")
        # open TextCtrl's edited crop
        edited_img = Image.open(SMOKE_OUT + "/" + name).convert("RGB")
        # column 1: draw the source crop
        axes[row][0].imshow(source_img)
        # column 2: write the word it shows now and the word we asked for
        axes[row][1].text(0.5, 0.5, source_words.get(name, "?") + "  ->  " + target_words.get(name, "?"), ha="center", va="center", fontsize=11)
        # column 3: draw TextCtrl's result
        axes[row][2].imshow(edited_img)
        # go through the three columns of this row
        for col in range(3):
            # hide the axis numbers (pixel positions are not useful here)
            axes[row][col].axis("off")
    # name column 1 on the top row
    axes[0][0].set_title("source crop", fontsize=10)
    # name column 2 on the top row
    axes[0][1].set_title("source text -> target text", fontsize=10)
    # name column 3 on the top row
    axes[0][2].set_title("TextCtrl result", fontsize=10)
    # give the whole figure a title
    fig.suptitle("TextCtrl smoke test on the authors' example crops", fontsize=11)
    # tidy the spacing
    fig.tight_layout()
    # save the picture as a PNG so it can go on a slide
    fig.savefig(FIGURE_PATH, dpi=150)
    # show it in the notebook
    plt.show()
    # say where the grid was saved
    print("Figure saved to:", FIGURE_PATH)